# ClickLens — Train All Models (Colab)

**Only 1 file to upload:** `features.csv`

Thumbnails are downloaded automatically from YouTube. Set runtime to **GPU** (Runtime > Change runtime type > T4 GPU), then Run All.

In [1]:
!pip install -q xgboost timm seaborn

In [2]:
from google.colab import files as colab_files
from pathlib import Path
import pandas as pd

P = Path("/content/ClickLens")
for d in ["data/processed","data/raw","models","data/outputs"]:
    (P/d).mkdir(parents=True, exist_ok=True)

CSV = P/"data/processed/features.csv"
if not CSV.exists():
    print("Upload features.csv:")
    up = colab_files.upload()
    for n,d in up.items(): (CSV).write_bytes(d)

df = pd.read_csv(CSV)
print(f"Loaded {len(df)} samples | Labels: {df['label'].value_counts().to_dict()} | Niches: {list(df['niche'].unique())}")

Upload features.csv:


Saving features.csv to features.csv
Loaded 3480 samples | Labels: {'Medium': 1160, 'Low': 1160, 'High': 1160} | Niches: ['gaming', 'travel', 'fitness']


In [3]:
# Download thumbnails from YouTube
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

RAW = P/"data/raw"
for n in df["niche"].unique(): (RAW/n).mkdir(exist_ok=True)

def dl(row):
    v,n = row["video_id"], row["niche"]
    lp = RAW/n/f"{v}.jpg"
    if lp.exists(): return str(lp), True
    for q in ["maxresdefault","hqdefault","mqdefault"]:
        try:
            r = requests.get(f"https://img.youtube.com/vi/{v}/{q}.jpg", timeout=10)
            if r.status_code==200 and len(r.content)>1000:
                lp.write_bytes(r.content); return str(lp), True
        except: continue
    return str(lp), False

paths = {}; ok=0; fail=0
with ThreadPoolExecutor(max_workers=20) as ex:
    fs = {ex.submit(dl, row): i for i,row in df.iterrows()}
    for i,f in enumerate(as_completed(fs)):
        p,s = f.result(); paths[fs[f]]=p
        if s: ok+=1
        else: fail+=1
        if (i+1)%500==0: print(f"  {i+1}/{len(df)}")

df["thumbnail_path"] = df.index.map(paths)
print(f"Done: {ok} ok, {fail} failed")
if fail>0:
    bf=len(df); df=df[df["thumbnail_path"].apply(lambda x: Path(x).exists())].reset_index(drop=True)
    print(f"Dropped {bf-len(df)} missing")

  500/3480
  1000/3480
  1500/3480
  2000/3480
  2500/3480
  3000/3480
Done: 3480 ok, 0 failed


In [4]:
# Common imports and constants
import json, pickle
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np, seaborn as sns, timm
import torch, torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from xgboost import XGBClassifier

SEED=42; FCOLS=[
  "dominant_color_1_r","dominant_color_1_g","dominant_color_1_b",
  "dominant_color_2_r","dominant_color_2_g","dominant_color_2_b",
  "dominant_color_3_r","dominant_color_3_g","dominant_color_3_b",
  "brightness","contrast","saturation","face_count","face_area_ratio",
  "text_present","text_area_ratio","edge_density","color_entropy"]
LMAP={"Low":0,"Medium":1,"High":2}; CNAMES=["Low","Medium","High"]; NC=3
IMEAN=[0.485,0.456,0.406]; ISTD=[0.229,0.224,0.225]
MDIR=P/"models"; ODIR=P/"data/outputs"
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")

PyTorch 2.10.0+cu128 | CUDA: True


## 1. Baseline (Majority Class)

In [5]:
tr_bl, te_bl = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df["label"])
maj = tr_bl["label"].value_counts().idxmax()
y_te_bl = te_bl["label"].values; y_pr_bl = [maj]*len(y_te_bl)
print(f"Majority class: {maj}")
print(f"Accuracy: {accuracy_score(y_te_bl,y_pr_bl):.4f}")
print(f"F1 macro: {f1_score(y_te_bl,y_pr_bl,average='macro',zero_division=0):.4f}")
print(f"Precision: {precision_score(y_te_bl,y_pr_bl,average='macro',zero_division=0):.4f}")
print(f"Recall: {recall_score(y_te_bl,y_pr_bl,average='macro',zero_division=0):.4f}")
with open(MDIR/"baseline_model.pkl","wb") as f:
    pickle.dump({"majority_class":maj,"class_distribution":tr_bl["label"].value_counts().to_dict()},f)
print("Saved baseline_model.pkl")

Majority class: Medium
Accuracy: 0.3333
F1 macro: 0.1667
Precision: 0.1111
Recall: 0.3333
Saved baseline_model.pkl


## 2. XGBoost

In [6]:
le = LabelEncoder(); le.classes_=np.array(["Low","Medium","High"])
df["label_enc"]=le.transform(df["label"])
X=df[FCOLS].values; y=df["label_enc"].values
Xtv,Xte,ytv,yte = train_test_split(X,y,test_size=0.15,random_state=SEED,stratify=y)
Xtr,Xv,ytr,yv = train_test_split(Xtv,ytv,test_size=0.15/0.85,random_state=SEED,stratify=ytv)
print(f"Train:{len(Xtr)} Val:{len(Xv)} Test:{len(Xte)}")

gs = GridSearchCV(
    XGBClassifier(objective="multi:softprob",num_class=3,eval_metric="mlogloss",use_label_encoder=False,random_state=SEED,verbosity=0),
    {"max_depth":[3,5,7],"learning_rate":[0.01,0.1,0.3],"n_estimators":[100,200],"subsample":[0.8,1.0]},
    scoring="f1_macro",cv=3,n_jobs=-1,verbose=1)
print("Grid search ...")
gs.fit(Xtr,ytr)
print(f"Best: {gs.best_params_} | CV F1: {gs.best_score_:.4f}")

Train:2436 Val:522 Test:522
Grid search ...
Fitting 3 folds for each of 36 candidates, totalling 108 fits
Best: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 200, 'subsample': 1.0} | CV F1: 0.4994


In [7]:
xgb = XGBClassifier(**gs.best_params_,objective="multi:softprob",num_class=3,eval_metric="mlogloss",
                     use_label_encoder=False,random_state=SEED,verbosity=0)
xgb.fit(Xtr,ytr)
yp_xgb = xgb.predict(Xte)
print(f"Accuracy: {accuracy_score(yte,yp_xgb):.4f}")
print(f"F1 macro: {f1_score(yte,yp_xgb,average='macro'):.4f}")
print(f"Precision: {precision_score(yte,yp_xgb,average='macro'):.4f}")
print(f"Recall: {recall_score(yte,yp_xgb,average='macro'):.4f}")

# Feature importance
imp=xgb.feature_importances_; si=np.argsort(imp)
fig,ax=plt.subplots(figsize=(10,8))
ax.barh([FCOLS[i] for i in si],imp[si],color="steelblue")
ax.set_xlabel("Importance"); ax.set_title("XGBoost Feature Importances")
plt.tight_layout(); fig.savefig(ODIR/"xgboost_feature_importances.png",dpi=150); plt.show()

with open(MDIR/"xgboost_model.pkl","wb") as f: pickle.dump(xgb,f)
with open(MDIR/"label_encoder.pkl","wb") as f: pickle.dump(le,f)
print("Saved xgboost_model.pkl + label_encoder.pkl")

Accuracy: 0.4808
F1 macro: 0.4814
Precision: 0.4820
Recall: 0.4808
Saved xgboost_model.pkl + label_encoder.pkl


## 3. EfficientNet-B0

In [8]:
class ThumbDS(Dataset):
    def __init__(s,d,t): s.df=d.reset_index(drop=True); s.t=t
    def __len__(s): return len(s.df)
    def __getitem__(s,i):
        r=s.df.iloc[i]; p=Path(r["thumbnail_path"])
        if not p.is_absolute(): p=P/p
        return s.t(Image.open(p).convert("RGB")), LMAP[r["label"]]

trT=transforms.Compose([transforms.RandomResizedCrop(224),transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2,0.2,0.2,0.1),transforms.ToTensor(),transforms.Normalize(IMEAN,ISTD)])
evT=transforms.Compose([transforms.Resize(256),transforms.CenterCrop(224),
    transforms.ToTensor(),transforms.Normalize(IMEAN,ISTD)])

BS=32; NW=2; NEP=30; PAT=5; LR=1e-4
df["_sk"]=df["label"]+"_"+df["niche"].astype(str)
tv,ted=train_test_split(df,test_size=0.15,random_state=SEED,stratify=df["_sk"])
trd,vd=train_test_split(tv,test_size=0.15/0.85,random_state=SEED,stratify=tv["_sk"])
print(f"Train:{len(trd)} Val:{len(vd)} Test:{len(ted)}")
trL=DataLoader(ThumbDS(trd,trT),batch_size=BS,shuffle=True,num_workers=NW,pin_memory=True)
vaL=DataLoader(ThumbDS(vd,evT),batch_size=BS,shuffle=False,num_workers=NW,pin_memory=True)
teL=DataLoader(ThumbDS(ted,evT),batch_size=BS,shuffle=False,num_workers=NW,pin_memory=True)

Train:2436 Val:522 Test:522


In [9]:
dev=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {dev}")
m=timm.create_model("efficientnet_b0",pretrained=True)
for p in m.parameters(): p.requires_grad=False
for b in m.blocks[-2:]:
    for p in b.parameters(): p.requires_grad=True
inf=m.classifier.in_features
m.classifier=nn.Sequential(nn.Linear(inf,256),nn.ReLU(),nn.Dropout(0.3),nn.Linear(256,NC))
m=m.to(dev)
print(f"Trainable: {sum(p.numel() for p in m.parameters() if p.requires_grad):,} / {sum(p.numel() for p in m.parameters()):,}")

Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Trainable: 3,072,287 / 4,336,255


In [11]:
def tr1(m,ld,cr,op,dv):
    m.train(); rl=c=t=0
    for im,lb in ld:
        im,lb=im.to(dv),lb.to(dv); op.zero_grad()
        o=m(im); l=cr(o,lb); l.backward(); op.step()
        rl+=l.item()*im.size(0); c+=o.max(1)[1].eq(lb).sum().item(); t+=lb.size(0)
    return rl/t, c/t

@torch.no_grad()
def ev(m,ld,cr,dv):
    m.eval(); rl=0; ap=[]; al=[]
    for im,lb in ld:
        im,lb=im.to(dv),lb.to(dv); o=m(im); l=cr(o,lb)
        rl+=l.item()*im.size(0); ap.extend(o.max(1)[1].cpu().numpy()); al.extend(lb.cpu().numpy())
    return rl/len(al), accuracy_score(al,ap), np.array(ap), np.array(al)

BMP=MDIR/"efficientnet_best.pth"
opt=torch.optim.Adam(filter(lambda p:p.requires_grad,m.parameters()),lr=LR)
sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="min",factor=0.5,patience=2)
crit=nn.CrossEntropyLoss()
hist={"train_loss":[],"train_acc":[],"val_loss":[],"val_acc":[]}
bvl=float("inf"); eni=0

for ep in range(1,NEP+1):
    tl,ta=tr1(m,trL,crit,opt,dev)
    vl,va,_,_=ev(m,vaL,crit,dev)
    sch.step(vl)
    hist["train_loss"].append(tl); hist["train_acc"].append(ta)
    hist["val_loss"].append(vl); hist["val_acc"].append(va)
    print(f"Epoch {ep:02d}/{NEP} tl={tl:.4f} ta={ta:.4f} vl={vl:.4f} va={va:.4f}")
    if vl<bvl:
        bvl=vl; eni=0; torch.save(m.state_dict(),BMP); print(f"  -> saved (vl={vl:.4f})")
    else:
        eni+=1
        if eni>=PAT: print(f"Early stop at epoch {ep}"); break

with open(MDIR/"training_history.json","w") as f: json.dump(hist,f,indent=2)
print("Training complete.")

Epoch 01/30 tl=1.0847 ta=0.4039 vl=1.0572 va=0.4617
  -> saved (vl=1.0572)
Epoch 02/30 tl=1.0258 ta=0.4856 vl=0.9995 va=0.5000
  -> saved (vl=0.9995)
Epoch 03/30 tl=0.9641 ta=0.5283 vl=0.9838 va=0.4943
  -> saved (vl=0.9838)
Epoch 04/30 tl=0.9111 ta=0.5722 vl=0.9423 va=0.5383
  -> saved (vl=0.9423)
Epoch 05/30 tl=0.8490 ta=0.6043 vl=0.9273 va=0.5613
  -> saved (vl=0.9273)
Epoch 06/30 tl=0.7958 ta=0.6346 vl=0.9259 va=0.5575
  -> saved (vl=0.9259)
Epoch 07/30 tl=0.7599 ta=0.6597 vl=0.9299 va=0.5785
Epoch 08/30 tl=0.7128 ta=0.6921 vl=0.9346 va=0.5958
Epoch 09/30 tl=0.6687 ta=0.7094 vl=0.9791 va=0.5594
Epoch 10/30 tl=0.6037 ta=0.7401 vl=0.9476 va=0.5939
Epoch 11/30 tl=0.5955 ta=0.7529 vl=0.9672 va=0.6034
Early stop at epoch 11
Training complete.


In [12]:
m.load_state_dict(torch.load(BMP,map_location=dev,weights_only=True))
_,tacc,eff_yp,eff_yt=ev(m,teL,crit,dev)
print(f"EfficientNet Test — Acc:{tacc:.4f} F1:{f1_score(eff_yt,eff_yp,average='macro'):.4f} "
      f"Prec:{precision_score(eff_yt,eff_yp,average='macro'):.4f} Rec:{recall_score(eff_yt,eff_yp,average='macro'):.4f}")

# Training curves
fig,(a1,a2)=plt.subplots(1,2,figsize=(14,5))
er=range(1,len(hist["train_loss"])+1)
a1.plot(er,hist["train_loss"],label="Train"); a1.plot(er,hist["val_loss"],label="Val")
a1.set_xlabel("Epoch"); a1.set_ylabel("Loss"); a1.set_title("Loss"); a1.legend()
a2.plot(er,hist["train_acc"],label="Train"); a2.plot(er,hist["val_acc"],label="Val")
a2.set_xlabel("Epoch"); a2.set_ylabel("Accuracy"); a2.set_title("Accuracy"); a2.legend()
plt.tight_layout(); fig.savefig(ODIR/"efficientnet_training_curves.png",dpi=150); plt.show()

EfficientNet Test — Acc:0.5613 F1:0.5598 Prec:0.5594 Rec:0.5613


## 4. Unified Evaluation

In [13]:
def cm(yt,yp): return {"accuracy":accuracy_score(yt,yp),"f1_macro":f1_score(yt,yp,average="macro",zero_division=0),
    "precision_macro":precision_score(yt,yp,average="macro",zero_division=0),"recall_macro":recall_score(yt,yp,average="macro",zero_division=0)}
def scm(yt,yp,cn,t,sp):
    c=confusion_matrix(yt,yp,labels=range(len(cn)))
    fig,ax=plt.subplots(figsize=(7,6))
    sns.heatmap(c,annot=True,fmt="d",cmap="Blues",xticklabels=cn,yticklabels=cn,ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(t)
    plt.tight_layout(); fig.savefig(sp,dpi=150); plt.show()

res={}
res["Baseline"]=cm(y_te_bl,y_pr_bl)
res["XGBoost"]=cm(yte,yp_xgb); scm(yte,yp_xgb,CNAMES,"XGBoost CM",ODIR/"confusion_matrix_xgboost.png")
res["EfficientNet"]=cm(eff_yt,eff_yp); scm(eff_yt,eff_yp,CNAMES,"EfficientNet CM",ODIR/"confusion_matrix_efficientnet.png")

cdf=pd.DataFrame(res).T; cdf.index.name="model"; cdf=cdf.reset_index()
cdf.to_csv(ODIR/"model_comparison.csv",index=False)
print("="*72)
print(f"{'Model':<18}{'Accuracy':>10}{'F1':>10}{'Precision':>10}{'Recall':>10}")
print("-"*72)
for _,r in cdf.iterrows():
    print(f"{r['model']:<18}{r['accuracy']:>10.4f}{r['f1_macro']:>10.4f}{r['precision_macro']:>10.4f}{r['recall_macro']:>10.4f}")
print("="*72)

Model               Accuracy        F1 Precision    Recall
------------------------------------------------------------------------
Baseline              0.3333    0.1667    0.1111    0.3333
XGBoost               0.4808    0.4814    0.4820    0.4808
EfficientNet          0.5613    0.5598    0.5594    0.5613


## 5. Cross-Niche Experiment

In [14]:
def mkx(): return XGBClassifier(max_depth=5,learning_rate=0.1,n_estimators=200,objective="multi:softprob",
    num_class=3,eval_metric="mlogloss",use_label_encoder=False,random_state=SEED,verbosity=0)

niches=sorted(df["niche"].unique()); rl=niches+["All"]
am=pd.DataFrame(np.nan,index=rl,columns=rl,dtype=float)
fm=pd.DataFrame(np.nan,index=rl,columns=rl,dtype=float)

for sn in niches:
    sd=df[df["niche"]==sn]; Xs=sd[FCOLS].values; ys=sd["label_enc"].values
    Xtr,Xts,ytr,yts=train_test_split(Xs,ys,test_size=0.2,random_state=SEED,stratify=ys)
    c=mkx(); c.fit(Xtr,ytr); yps=c.predict(Xts)
    am.loc[sn,sn]=accuracy_score(yts,yps); fm.loc[sn,sn]=f1_score(yts,yps,average="macro",zero_division=0)
    cf=mkx(); cf.fit(Xs,ys)
    for tn in niches:
        if tn==sn: continue
        td=df[df["niche"]==tn]; yp=cf.predict(td[FCOLS].values)
        am.loc[sn,tn]=accuracy_score(td["label_enc"].values,yp)
        fm.loc[sn,tn]=f1_score(td["label_enc"].values,yp,average="macro",zero_division=0)

Xa,ya=df[FCOLS].values,df["label_enc"].values
Xtra,Xtea,ytra,ytea=train_test_split(Xa,ya,test_size=0.2,random_state=SEED,stratify=ya)
ca=mkx(); ca.fit(Xtra,ytra)
am.loc["All","All"]=accuracy_score(ytea,ca.predict(Xtea))
fm.loc["All","All"]=f1_score(ytea,ca.predict(Xtea),average="macro",zero_division=0)
for n in niches:
    nd=df[df["niche"]==n]; yp=ca.predict(nd[FCOLS].values)
    am.loc["All",n]=accuracy_score(nd["label_enc"].values,yp)
    fm.loc["All",n]=f1_score(nd["label_enc"].values,yp,average="macro",zero_division=0)

print("Accuracy:"); print(am.round(4).to_string())
print("\nF1 Macro:"); print(fm.round(4).to_string())

# Save
af=am.stack().reset_index(); af.columns=["train","test","value"]; af["metric"]="accuracy"
ff=fm.stack().reset_index(); ff.columns=["train","test","value"]; ff["metric"]="f1_macro"
pd.concat([af,ff])[["metric","train","test","value"]].to_csv(ODIR/"cross_niche_matrix.csv",index=False)

fig,axes=plt.subplots(1,2,figsize=(14,5))
for ax,mx,t in [(axes[0],am,"Accuracy"),(axes[1],fm,"F1 macro")]:
    sns.heatmap(mx.astype(float),annot=True,fmt=".3f",cmap="YlOrRd",vmin=0,vmax=1,linewidths=0.5,ax=ax)
    ax.set_title(f"Cross-Niche {t}"); ax.set_xlabel("Test"); ax.set_ylabel("Train")
plt.suptitle("Does good thumbnail design transfer across niches?",fontsize=13,fontweight="bold",y=1.02)
plt.tight_layout(); fig.savefig(ODIR/"cross_niche_heatmap.png",dpi=150,bbox_inches="tight"); plt.show()

Accuracy:
         fitness  gaming  travel     All
fitness   0.6154  0.3572  0.3068     NaN
gaming    0.4081  0.5941  0.3183     NaN
travel    0.3025  0.3284  0.4426     NaN
All       0.8878  0.8928  0.8556  0.5029

F1 Macro:
         fitness  gaming  travel     All
fitness   0.6076  0.3410  0.2857     NaN
gaming    0.3788  0.5823  0.3175     NaN
travel    0.3003  0.3206  0.4343     NaN
All       0.8844  0.8916  0.8549  0.5037


## 6. Download Artifacts

In [15]:
import zipfile
zp="/content/clicklens_artifacts.zip"
with zipfile.ZipFile(zp,"w",zipfile.ZIP_DEFLATED) as z:
    for d,pf in [(MDIR,"models"),(ODIR,"data/outputs")]:
        for f in sorted(d.iterdir()):
            if f.is_file(): z.write(f,f"{pf}/{f.name}")
print(f"Zip: {Path(zp).stat().st_size/1024/1024:.1f} MB")
print("\nContents:")
for d,pf in [(MDIR,"models"),(ODIR,"data/outputs")]:
    for f in sorted(d.iterdir()):
        if f.is_file():
            s=f.stat().st_size; print(f"  {pf}/{f.name} ({s/1024:.0f} KB)")

print("\nDownloading zip...")
colab_files.download(zp)

Zip: 16.6 MB

Contents:
  models/baseline_model.pkl (0 KB)
  models/efficientnet_best.pth (17231 KB)
  models/label_encoder.pkl (0 KB)
  models/training_history.json (1 KB)
  models/xgboost_model.pkl (2194 KB)
  data/outputs/confusion_matrix_efficientnet.png (36 KB)
  data/outputs/confusion_matrix_xgboost.png (36 KB)
  data/outputs/cross_niche_heatmap.png (83 KB)
  data/outputs/cross_niche_matrix.csv (1 KB)
  data/outputs/efficientnet_training_curves.png (90 KB)
  data/outputs/model_comparison.csv (0 KB)
  data/outputs/xgboost_feature_importances.png (78 KB)



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**After download**, unzip in your project root:
```bash
cd ClickLens/ && unzip clicklens_artifacts.zip
```